In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [2]:
users = pd.read_csv('takehome_users.csv', encoding='latin-1')
engagement = pd.read_csv('takehome_user_engagement.csv')

users.head()

,object_id,creation_time,name,email,creation_source,last_session_creation_time,opted_in_to_mailing_list,enabled_for_marketing_drip,org_id,invited_by_user_id
0,1,2014-04-22 03:53:30,Clausen August,AugustCClausen@yahoo.com,GUEST_INVITE,1.398139e+09,1,0,11,10803.0
1,2,2013-11-15 03:45:04,Poole Matthew,MatthewPoole@gustr.com,ORG_INVITE,1.396238e+09,0,0,1,316.0
2,3,2013-03-19 23:14:52,Bottrill Mitchell,MitchellBottrill@gustr.com,ORG_INVITE,1.363735e+09,0,0,94,1525.0
3,4,2013-05-21 08:09:28,Clausen Nicklas,NicklasSClausen@yahoo.com,GUEST_INVITE,1.369210e+09,0,0,1,5151.0
4,5,2013-01-17 10:14:20,Raw Grace,GraceRaw@yahoo.com,GUEST_INVITE,1.358850e+09,0,0,193,5240.0


In [3]:
engagement.head()

,time_stamp,user_id,visited
0,2014-04-22 03:53:30,1,1
1,2013-11-15 03:45:04,2,1
2,2013-11-29 03:45:04,2,1
3,2013-12-09 03:45:04,2,1
4,2013-12-25 03:45:04,2,1


In [4]:
#create target variable "adopted" 
engagement["time_stamp"] = pd.to_datetime(engagement["time_stamp"])
engagement["date"] = engagement["time_stamp"].dt.date

# one row per user per login day
daily_logins = (
    engagement[["user_id", "date"]]
    .drop_duplicates()
    .sort_values(["user_id", "date"])
)

def is_adopted(dates):
    dates = pd.to_datetime(dates).sort_values().reset_index(drop=True)
    for i in range(len(dates) - 2):
        if (dates.iloc[i + 2] - dates.iloc[i]).days <= 6:
            return 1
    return 0

adopted = (
    daily_logins.groupby("user_id")["date"]
    .apply(is_adopted)
    .reset_index(name="adopted_user")
)

In [5]:
adopted.head()

,user_id,adopted_user
0,1,0
1,2,1
2,3,0
3,4,0
4,5,0


In [6]:
#merge adopted with users into one df
users = users.rename(columns={"object_id": "user_id"})
df = users.merge(adopted, on="user_id", how="left")
df["adopted_user"] = df["adopted_user"].fillna(0).astype(int)

df.head()

,user_id,creation_time,name,email,creation_source,last_session_creation_time,opted_in_to_mailing_list,enabled_for_marketing_drip,org_id,invited_by_user_id,adopted_user
0,1,2014-04-22 03:53:30,Clausen August,AugustCClausen@yahoo.com,GUEST_INVITE,1.398139e+09,1,0,11,10803.0,0
1,2,2013-11-15 03:45:04,Poole Matthew,MatthewPoole@gustr.com,ORG_INVITE,1.396238e+09,0,0,1,316.0,1
2,3,2013-03-19 23:14:52,Bottrill Mitchell,MitchellBottrill@gustr.com,ORG_INVITE,1.363735e+09,0,0,94,1525.0,0
3,4,2013-05-21 08:09:28,Clausen Nicklas,NicklasSClausen@yahoo.com,GUEST_INVITE,1.369210e+09,0,0,1,5151.0,0
4,5,2013-01-17 10:14:20,Raw Grace,GraceRaw@yahoo.com,GUEST_INVITE,1.358850e+09,0,0,193,5240.0,0


In [7]:
#feature engineering
df["creation_time"] = pd.to_datetime(df["creation_time"])
df["last_session_creation_time"] = pd.to_datetime(
    df["last_session_creation_time"], unit="s"
)

df["has_inviter"] = df["invited_by_user_id"].notna().astype(int)
df["email_domain"] = df["email"].str.split("@").str[-1]

# Keep only common domains, group the rest
top_domains = df["email_domain"].value_counts().head(10).index
df["email_domain_grouped"] = np.where(
    df["email_domain"].isin(top_domains),
    df["email_domain"],
    "other"
)

df["account_age_days"] = (
    df["last_session_creation_time"].max() - df["creation_time"]
).dt.days

df.head()

,user_id,creation_time,name,email,creation_source,last_session_creation_time,opted_in_to_mailing_list,enabled_for_marketing_drip,org_id,invited_by_user_id,adopted_user,has_inviter,email_domain,email_domain_grouped,account_age_days
0,1,2014-04-22 03:53:30,Clausen August,AugustCClausen@yahoo.com,GUEST_INVITE,2014-04-22 03:53:30,1,0,11,10803.0,0,1,yahoo.com,yahoo.com,45
1,2,2013-11-15 03:45:04,Poole Matthew,MatthewPoole@gustr.com,ORG_INVITE,2014-03-31 03:45:04,0,0,1,316.0,1,1,gustr.com,gustr.com,203
2,3,2013-03-19 23:14:52,Bottrill Mitchell,MitchellBottrill@gustr.com,ORG_INVITE,2013-03-19 23:14:52,0,0,94,1525.0,0,1,gustr.com,gustr.com,443
3,4,2013-05-21 08:09:28,Clausen Nicklas,NicklasSClausen@yahoo.com,GUEST_INVITE,2013-05-22 08:09:28,0,0,1,5151.0,0,1,yahoo.com,yahoo.com,381
4,5,2013-01-17 10:14:20,Raw Grace,GraceRaw@yahoo.com,GUEST_INVITE,2013-01-22 10:14:20,0,0,193,5240.0,0,1,yahoo.com,yahoo.com,505


In [8]:
#summary tables

adoption_rate = df["adopted_user"].mean()
print(f"Adoption rate: {adoption_rate:.2%}")

summary_creation = (
    df.groupby("creation_source")["adopted_user"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
)

summary_mailing = (
    df.groupby("opted_in_to_mailing_list")["adopted_user"]
    .agg(["mean", "count"])
)

summary_drip = (
    df.groupby("enabled_for_marketing_drip")["adopted_user"]
    .agg(["mean", "count"])
)

summary_inviter = (
    df.groupby("has_inviter")["adopted_user"]
    .agg(["mean", "count"])
)

summary_creation, summary_mailing, summary_drip, summary_inviter

Adoption rate: 13.35%


(                        mean  count
 creation_source                    
 SIGNUP_GOOGLE_AUTH  0.167509   1385
 GUEST_INVITE        0.166436   2163
 SIGNUP              0.140393   2087
 ORG_INVITE          0.129995   4254
 PERSONAL_PROJECTS   0.077688   2111,
                               mean  count
 opted_in_to_mailing_list                 
 0                         0.131912   9006
 1                         0.138277   2994,
                                 mean  count
 enabled_for_marketing_drip                 
 0                           0.132837  10208
 1                           0.137277   1792,
                  mean  count
 has_inviter                 
 0            0.123410   5583
 1            0.142278   6417)

In [9]:
features = [
    "creation_source",
    "opted_in_to_mailing_list",
    "enabled_for_marketing_drip",
    "org_id",
    "has_inviter",
    "email_domain_grouped",
    "account_age_days"
]

X = df[features].copy()
y = df["adopted_user"]

categorical_cols = ["creation_source", "org_id", "email_domain_grouped"]
numeric_cols = [
    "opted_in_to_mailing_list",
    "enabled_for_marketing_drip",
    "has_inviter",
    "account_age_days"
]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ("num", StandardScaler(), numeric_cols)
])

In [ ]:
# Logistic Regression Baseline

model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model.fit(X_train, y_train)

preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, preds))
print("ROC-AUC:", roc_auc_score(y_test, probs))

              precision    recall  f1-score   support

           0       0.87      1.00      0.93      2600
           1       0.50      0.01      0.01       400

    accuracy                           0.87      3000
   macro avg       0.68      0.50      0.47      3000
weighted avg       0.82      0.87      0.81      3000

ROC-AUC: 0.6055153846153846


In [10]:
# Random Forest
model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model.fit(X_train, y_train)

preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, preds))
print("ROC-AUC:", roc_auc_score(y_test, probs))

              precision    recall  f1-score   support

           0       0.87      0.98      0.92      2600
           1       0.16      0.02      0.04       400

    accuracy                           0.85      3000
   macro avg       0.51      0.50      0.48      3000
weighted avg       0.77      0.85      0.80      3000

ROC-AUC: 0.5982682692307693
